# Exercise: make your data tidy

In the previous notebook two split-apply-combine questions failed on
`predimed_2.csv`. The reason is that the table is not **tidy**.

Data is *tidy* when:

1. each **variable** forms its own column,
2. each **observation** forms its own row,
3. each type of observational unit forms its own table.

Let us load the data again and see how it breaks rule 1.

In [1]:
import pandas as pd

In [2]:
df = pd.read_csv('predimed_2.csv')
df.head()

,patient-id,group,sex_age,smoke_never,smoke_former,smoke_current,event
0,1325,MedDiet + VOO,Male_72,0,0,1,0
1,1326,Control,Female_69,1,0,0,0
2,1327,MedDiet + Nuts,Female_64,0,1,0,0
3,1328,Control,Female_70,1,0,0,0
4,1329,MedDiet + VOO,Male_65,0,1,0,0


This table breaks rule 1 in **two** different ways:

* `sex_age` holds **two** variables in one column (sex and age).
* the smoking status, a **single** variable, is split across **three** columns,
  `smoke_never`, `smoke_former` and `smoke_current`. 

We fix them one at a time.

## Problem 1: two variables in one column -> split

- Split `sex_age` into two separate columns, `sex` and `age`. The two pieces are
joined by an underscore, so we split on `'_'`.

Hint: `Series.str.split('_', expand=True)` returns one column per piece.

Note that after a string split every piece is text, so we need to convert `age` back to an integer.

In [3]:
# solution
df[['sex', 'age']] = df['sex_age'].str.split('_', expand=True)
df

,patient-id,group,sex_age,smoke_never,smoke_former,smoke_current,event,sex,age
0,1325,MedDiet + VOO,Male_72,0,0,1,0,Male,72
1,1326,Control,Female_69,1,0,0,0,Female,69
2,1327,MedDiet + Nuts,Female_64,0,1,0,0,Female,64
3,1328,Control,Female_70,1,0,0,0,Female,70
4,1329,MedDiet + VOO,Male_65,0,1,0,0,Male,65
...,...,...,...,...,...,...,...,...,...
395,1720,MedDiet + VOO,Male_67,0,0,1,0,Male,67
396,1721,Control,Male_68,0,1,0,0,Male,68
397,1722,MedDiet + VOO,Female_61,1,0,0,0,Female,61
398,1723,MedDiet + VOO,Female_72,1,0,0,0,Female,72


In [4]:
df['age'] = df['age'].astype(int)
df.head()

,patient-id,group,sex_age,smoke_never,smoke_former,smoke_current,event,sex,age
0,1325,MedDiet + VOO,Male_72,0,0,1,0,Male,72
1,1326,Control,Female_69,1,0,0,0,Female,69
2,1327,MedDiet + Nuts,Female_64,0,1,0,0,Female,64
3,1328,Control,Female_70,1,0,0,0,Female,70
4,1329,MedDiet + VOO,Male_65,0,1,0,0,Male,65


Now `age` is its own column, so we can calculate what we wanted:

In [5]:
# solution - mean age per diet group
df.groupby('group')['age'].mean()

group
Control           67.255034
MedDiet + Nuts    66.829060
MedDiet + VOO     67.529851
Name: age, dtype: float64

In [6]:
# solution - and, for free, events per sex
df.groupby('sex')['event'].sum()

sex
Female     6
Male      14
Name: event, dtype: int64

## Problem 2: one variable across several columns -> melt

- Convert the three (indicator) columns `smoke_never`, `smoke_former` and `smoke_current` into a single `smoke` column.

Hint: `DataFrame.melt(id_vars=..., value_vars=..., var_name=..., value_name=...)`.

In [7]:
# solution:
df_tidy = df.melt(
    id_vars=['patient-id', 'group', 'sex', 'age', 'event'],
    value_vars=['smoke_never', 'smoke_former', 'smoke_current'],
    var_name='smoke',
    value_name='indicator',
)
df_tidy

,patient-id,group,sex,age,event,smoke,indicator
0,1325,MedDiet + VOO,Male,72,0,smoke_never,0
1,1326,Control,Female,69,0,smoke_never,1
2,1327,MedDiet + Nuts,Female,64,0,smoke_never,0
3,1328,Control,Female,70,0,smoke_never,1
4,1329,MedDiet + VOO,Male,65,0,smoke_never,0
...,...,...,...,...,...,...,...
1195,1720,MedDiet + VOO,Male,67,0,smoke_current,1
1196,1721,Control,Male,68,0,smoke_current,0
1197,1722,MedDiet + VOO,Female,61,0,smoke_current,0
1198,1723,MedDiet + VOO,Female,72,0,smoke_current,0


For each patient exactly one of the three indicators is 1, so after melting we keep only the rows where the indicator equals 1, and then clean up the label (`smoke_never` -> `never`).


In [8]:
# keep only the row that marks each patient's actual status, then drop the helper column
# solution:
df_tidy = df_tidy[df_tidy['indicator'] == 1].drop(columns='indicator')
df_tidy

,patient-id,group,sex,age,event,smoke
1,1326,Control,Female,69,0,smoke_never
3,1328,Control,Female,70,0,smoke_never
6,1331,Control,Female,70,0,smoke_never
8,1333,MedDiet + Nuts,Female,69,0,smoke_never
9,1334,Control,Male,70,0,smoke_never
...,...,...,...,...,...,...
1178,1703,Control,Male,71,0,smoke_current
1183,1708,MedDiet + VOO,Male,52,0,smoke_current
1186,1711,Control,Female,80,0,smoke_current
1194,1719,MedDiet + VOO,Male,56,0,smoke_current


In [9]:
# optional clean up o the labels: 'smoke_never' -> 'never'
df_tidy['smoke'] = df_tidy['smoke'].str.replace('smoke_', '')
df_tidy.head()

,patient-id,group,sex,age,event,smoke
1,1326,Control,Female,69,0,never
3,1328,Control,Female,70,0,never
6,1331,Control,Female,70,0,never
8,1333,MedDiet + Nuts,Female,69,0,never
9,1334,Control,Male,70,0,never


Now we  can answer what we wanted: how many events happened per smoking status

In [10]:
# events per smoking status
df_tidy.groupby('smoke')['event'].sum()

smoke
current    7
former     8
never      5
Name: event, dtype: int64

And because the table is now fully tidy, we can combine variables freely, for
example the number of events per diet group and smoking status:

In [11]:
# 
df_tidy.pivot_table(index='group', columns='smoke', values='event', aggfunc='sum')

smoke,current,former,never
group,,,
Control,2,4,3
MedDiet + Nuts,2,2,1
MedDiet + VOO,3,2,1
